# Step 1 · Meet the model: tokens, numbers and next-word guesses

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 30 minutes · ⚡ GPU recommended (in Colab: **Runtime → Change runtime type → T4 GPU**)

> ### 🤔 Wonder
> If you type *"The hot stove felt"*, what does an AI language model expect to come next? And what does that "expectation" look like inside the machine?

**In this step you will:**

1. Load a real, openly available language model and read its "table of contents".
2. See how text is chopped into **tokens** and turned into ID numbers.
3. Run the model once and inspect **everything it hands back**.
4. Turn the model's raw output into next-word probabilities.
5. Find the **hidden numbers** (activations) that the rest of the lab is about.

New to the signposts (📦 🔍 🎛️ 🧱 🛠️)? Do [Step 0](00_how_to_read_code.ipynb) first; it takes 25 minutes and makes everything below easier.

### How a language model works, in five ideas

**① It predicts the next piece of text.** A language model is a very powerful autocomplete. Given *"The warm blanket felt"*, it scores every possible next piece of text. To write, it picks one, adds it, and repeats.

**② Text is chopped into tokens.** A **token** is a chunk of text: a word, part of a word, or punctuation. Each token has an ID number.

**③ Each token becomes a long list of numbers (a vector).** In our model, every token is represented by **1,536 numbers** at every stage of processing.

**④ The numbers pass through a stack of layers.** Our model has **28 layers**. Each layer reads the current numbers and adds its own adjustments, like 28 editors each revising the same shared note. That running note is called the **residual stream**.

**⑤ Weights vs. activations.** **Weights** are the model's learned settings, fixed before this lab: the recipe. **Activations** are the numbers computed for one particular input: the dish cooked today. Different sentences give different activations; the recipe never changes. **In this lab we never change the weights.**

## Part 1 · Set up and load the model

Run these two cells in order. The second downloads the model (about 3 GB the first time, which can take a few minutes).

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

### 📦 What came back?

`pclab.load_model(...)` returned **two** things, which we unpacked into two variables:

- `tokenizer`: turns text into token IDs and back.
- `model`: the neural network itself.

Let's look at both, and at a few facts worth recording with any result.

In [ ]:
look(tokenizer, 'tokenizer')
look(model, 'model')
facts = pclab.model_facts(model, MODEL_NAME)
look(facts, 'facts')

### 🔍 Open the box: how the model was loaded

`load_model` is a short function. Read it, then the walk-through.

In [ ]:
show_source(pclab.load_model)

### 🔍 Code walk-through: `load_model`

| Code | What it does | Why it's written this way |
|---|---|---|
| `model_name=DEFAULT_MODEL` | names the model on the Hugging Face Hub; `DEFAULT_MODEL` is a constant set to `'Qwen/Qwen2.5-1.5B'` | a free, openly licensed (Apache 2.0) model, small enough for a free Colab GPU |
| `revision='main'` | which saved version of the model to download | models get updated; recording the exact version (shown in `facts['resolved_revision']`) lets others repeat your work |
| `dtype=torch.float32` | stores numbers with 32 bits | 🎛️ 16-bit formats halve the memory. float16 can overflow in this model family; bfloat16 is safe but less precise. Our experiment is small, so we use float32 for simplicity and precision. |
| the `PCLAB_TEST_MODEL` line | swaps in a tiny random model | only used by the lab's automatic tests; ignore it |
| `'cuda' if torch.cuda.is_available() else 'cpu'` | uses the GPU if there is one | GPUs do the model's math much faster |
| `AutoTokenizer.from_pretrained(...)` | downloads and builds the matching tokenizer | a model only understands the token IDs it was trained with, so the tokenizer must match the model |
| `AutoModelForCausalLM.from_pretrained(...)` | downloads the weights and builds the network | "Causal LM" = a model that predicts the next token from the tokens before it |
| `model.to(device)` | moves the weights to the GPU (or CPU) | the model and its inputs must be on the same device |
| `model.eval()` | switches to "use" mode | turns off training-only behaviour (such as *dropout*, which randomly silences parts of the network during training) |
| `tokenizer.pad_token = tokenizer.eos_token` | sets a "padding" token if the model has none | some functions expect one; it never affects our single-sentence inputs |
| `return tokenizer, model` | hands back two things at once (as a tuple) | which is why we wrote `tokenizer, model = ...` |

### Read the model's table of contents

`print(model)` lists the model's parts. It looks intimidating, but you only need to find four things.

**✏️ Predict first:** how many times do you think the same layer design is repeated?

In [ ]:
print(model)

**Find these in the printout:**

| Part | What to look for | Meaning |
|---|---|---|
| `embed_tokens` | `Embedding(151936, 1536)` | a lookup table: one row of 1,536 numbers for each of ~152,000 tokens. This is where text becomes numbers. |
| `layers` | `(0-27): 28 x Qwen2DecoderLayer` | the 28 layers. Each has `self_attn` (attention: lets a token pull in information from earlier tokens) and `mlp` (a feed-forward network that transforms it further). |
| `norm` | `Qwen2RMSNorm((1536,))` | a final rescaling of the numbers after the last layer |
| `lm_head` | `Linear(in_features=1536, out_features=151936)` | turns the final 1,536 numbers into one score per possible next token |

Notice the counting: Python numbers the layers **0 to 27**. Humans usually say layers **1 to 28**. The lab prints both when it matters.

## Part 2 · Tokens: how text becomes numbers

In [ ]:
# 🧪 Experiment: change the sentence and rerun
sentence = 'The warm blanket felt comfortable.'
enc = tokenizer(sentence, return_tensors='pt')
look(enc, 'enc')

### 📦 What came back?

`tokenizer(...)` returned a **BatchEncoding**, which behaves like a dictionary with two keys:

| Key | Shape | Meaning |
|---|---|---|
| `input_ids` | `(1, number of tokens)` | the token ID numbers |
| `attention_mask` | same shape | 1 = "pay attention to this token". It would contain 0s if we padded several sentences to the same length. Here it's all 1s. |

**Why `(1, ...)` and not just `(...)`?** Models are built to process many texts at once, called a **batch**. Even one sentence is wrapped as a batch of one. That first dimension is the **batch dimension**, and you'll see `[0, ...]` used later to unwrap it.

> **🎛️ Why `return_tensors='pt'`?** Without it you get plain Python lists. `'pt'` asks for **P**y**T**orch tensors, which is what the model needs.
>
> **🛠️ Tinker:** delete `, return_tensors='pt'` in the cell above, rerun it, and compare what `look` reports.

In [ ]:
# 🔧 Show each token and its ID side by side
ids = enc['input_ids'][0].tolist()          # [0] removes the batch dimension; .tolist() makes a Python list
pieces = tokenizer.convert_ids_to_tokens(ids)
display(pd.DataFrame({'token_id': ids, 'token_piece': pieces}))
print('Decoded back to text:', repr(tokenizer.decode(ids)))

Some pieces start with `Ġ`. It stands for **a space before the word**, so `Ġwarm` means " warm". The ID numbers are just labels, like locker numbers; a bigger ID doesn't mean anything.

**🧪 Try:** change the sentence to include *"uncomfortable"* or *"scalded"*. Is each one token or several? Why might rarer words be split into pieces?

## Part 3 · Run the model once and inspect everything it returns

`model(**enc)` runs the whole model on our tokens. Two pieces of syntax to decode:

- `**enc` **unpacks** the dictionary into named arguments. It's shorthand for `model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])`.
- `with torch.inference_mode():` tells PyTorch we're only *using* the model, not training it, so it can skip the bookkeeping training needs. Faster, and uses less memory.

In [ ]:
# 🔧 One forward pass
enc = tokenizer('The hot stove felt', return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    outputs = model(**enc, use_cache=False)

look(outputs, 'outputs')
look(outputs.logits, 'outputs.logits', dims=('texts in batch', 'tokens', 'vocabulary entries'))

### 📦 What came back?

`outputs` is a **model output object**: a dictionary-like container. Its main part is **`logits`**, shape `(1, tokens, 151936)`.

- `1`: one text in the batch.
- `tokens`: one row **for every position** in the input.
- `151936`: one score for **every entry in the vocabulary**.

**Why a row for every position, when we only want the *next* word?** The model was trained to predict the next token after *every* prefix at once: after "The", after "The hot", after "The hot stove", and so on. That's efficient for training. For us, only the **last row** matters: the prediction after the whole prompt.

**Logits are raw scores, not probabilities.** They can be negative and don't add up to 1. The **softmax** function converts them into probabilities.

> **🎛️ Why `use_cache=False`?** The model can keep a memory of earlier tokens (a "cache") to speed up writing long texts. We're running it once, so we switch that off and keep the output simpler.

In [ ]:
# 🔧 From the last row of logits to the top-10 next tokens
last_logits = outputs.logits[0, -1]                 # text 0, last position → one score per vocabulary entry
look(last_logits, 'last_logits', dims=('vocabulary entries',))

probs = torch.softmax(last_logits.float(), dim=-1)  # scores → probabilities that add up to 1
print('Probabilities add up to:', round(probs.sum().item(), 4))

top = torch.topk(probs, k=10)                       # the 10 largest probabilities and where they are
display(pd.DataFrame({
    'next_token': [tokenizer.decode([i]) for i in top.indices.tolist()],
    'probability': top.values.tolist(),
}))

### 🔍 Code walk-through

| Code | What it does | Why |
|---|---|---|
| `outputs.logits[0, -1]` | text 0, last position, all vocabulary scores | shape goes `(1, tokens, 151936)` → `(151936,)`. `-1` means "the last one", whatever the length. |
| `torch.softmax(..., dim=-1)` | turns scores into probabilities along the last dimension | `dim` in PyTorch means the same as `axis` in NumPy |
| `.float()` | makes sure we use 32-bit numbers | safe habit before softmax |
| `.item()` | turns a one-number tensor into a plain Python number | so it prints cleanly |
| `torch.topk(probs, k=10)` | finds the 10 largest values | returns **two** things: `.values` (the probabilities) and `.indices` (which token IDs they belong to) |
| `tokenizer.decode([i])` | turns one ID back into text | `decode` normally takes a list of IDs; here it's a one-item list |

### 🧪 Your turn: compare prompts

The cell below packages those steps into a function, so you can compare prompts quickly.

**✏️ Predict first:** for *"The hot stove felt"* vs. *"The warm bath felt"*, which next words do you expect near the top? Will " painful" or " comfortable" appear in the top 10 for either?

In [ ]:
# 🧪 Experiment: try your own prompts (write unfinished sentences, not questions)
@torch.inference_mode()
def top_next_tokens(prompt, k=10):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    logits = model(**enc, use_cache=False).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return pd.DataFrame({'next_token': [tokenizer.decode([i]) for i in top.indices.tolist()],
                         'probability': top.values.round(decimals=4).tolist()})

for prompt in ['The hot stove felt', 'The warm bath felt']:
    print(prompt, '…')
    display(top_next_tokens(prompt))

This is a **base model**: a pure autocomplete, not a chatbot. If you ask it a question it may simply continue your text. Write prompts as unfinished sentences.

## Part 4 · The hidden numbers: activations

The logits are the model's *final* answer. The rest of this lab is about the numbers **in between**: what each layer passes to the next. Ask for them with `output_hidden_states=True`.

**✏️ Predict first:** our model has 28 layers. How many sets of hidden numbers will come back?

In [ ]:
# 🔧 Ask the model for its hidden states (same prompt as before: 'The hot stove felt')
enc = tokenizer('The hot stove felt', return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    outputs = model(**enc, output_hidden_states=True, use_cache=False)

hidden_states = outputs.hidden_states
look(hidden_states, 'hidden_states')
print('Number of layers:', N_LAYERS, '| Number of hidden-state entries:', len(hidden_states))

### 📦 What came back?

A **tuple** with **one more entry than there are layers**:

- Entry `0` is the **embedding**: the numbers *before* any layer has worked on them (straight from the `embed_tokens` lookup table).
- Entry `k` (for `k` = 1 to 28) is the output of layer `k`, counting layers from 1. In Python's counting from 0, that's `LAYERS[k - 1]`.

Each entry has shape `(1, tokens, 1536)`: batch × position × numbers.

> **🧱 Why a tuple?** It's a fixed, ordered sequence, one item per stage, that nobody should accidentally change. A tuple can't be modified after it's created.

Now pull out **one vector**: the last token, halfway through the model.

In [ ]:
# 🧪 Experiment: change the stage (0 to N_LAYERS) and the position (-1 = last token, 0 = first)
stage = N_LAYERS // 2          # halfway: the output of layer 14 of 28 (Python index 13)
snapshot = hidden_states[stage][0, -1, :]
look(snapshot, 'snapshot', dims=('numbers',))

**That vector is a "snapshot"**: the model's internal state for the last token at one layer. Think of a mixing board with 1,536 sliders. No single slider is likely to mean "pain". If pain is represented at all, it's probably a *combination* of slider positions.

In Step 2 we'll take snapshots like this for 44 of the lab's 60 carefully written sentences (the rest wait, sealed, until Step 4). We'll use a more precise tool (a **hook**) instead of `output_hidden_states`, because it lets us grab exactly the numbers we want.

> **🔭 Going deeper:** the *last* entry of `hidden_states` has the final `norm` applied to it; the others don't. Step 2 checks this for itself.

### 🌱 A first curiosity: how big are the numbers at each layer?

Each snapshot is a list of 1,536 numbers. Its overall size (its **length**, or **norm**) is one number. Let's plot it across layers for our prompt.

**✏️ Predict first:** will the size stay about the same, shrink, or grow through the layers?

In [ ]:
# 🔧 Size of the last-token vector at every stage.
# We leave out the final entry: it has the final `norm` applied, so it isn't measured the same way.
stages = range(len(hidden_states) - 1)
sizes = [hidden_states[i][0, -1].float().norm().item() for i in stages]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(list(stages), sizes, 'o-', color=pclab.COLORS['accent'])
ax.set(xlabel='Stage (0 = embedding, k = after layer k)', ylabel='Vector length (norm)',
       title='How large is the last token\'s vector at each stage?')
plt.show()

Most models show the vector **growing** as layers add their adjustments to the shared note (the residual stream), often unevenly. This matters later: numbers at different layers live on very different scales, so "bigger scores" at a layer don't mean that layer is "more about pain". That's why Step 3 compares layers with **accuracy** and **AUC**, which only care about which side of the line, and in what order, sentences land, not how big the numbers are.

---

## 🧭 Explain it

1. Explain the difference between a **weight** and an **activation** using your own analogy.
2. What does each dimension of `outputs.logits` mean? Why does the model produce a row for every position?
3. Why does `hidden_states` have one more entry than the number of layers?
4. What is a **token**? Give one example from your experiments where a word became several tokens.

## 🌱 Curiosity branches

- **Words vs. world:** compare top next-tokens for *"The ice felt"*, *"The ice cube on the burn felt"*, *"The ice on the sidewalk felt"*. What context changes the prediction?
- **Tokens and rare words:** are pain words (*"scalded", "throbbing"*) split into more tokens than comfort words? Does it matter?
- **Norm growth:** does the vector-size curve look the same for a pain prompt and a comfort prompt?

**Next:** [Step 2 · Taking snapshots inside the model](02_taking_snapshots.ipynb)